# Sampling Methods -- DS Interview Reference

## Quick Index

| Goal | Method | Section |
| :--- | :--- | :--- |
| Split data into train/test without sklearn | Simple Random Sampling | §1 |
| Preserve class proportions in a split | Stratified Sampling | §2 |
| Sample efficiently from ordered/large data | Systematic Sampling | §3 |
| Sample grouped data (sessions, users) | Cluster Sampling | §4 |
| CI for any statistic without distributional assumptions | Bootstrap | §5 |
| Detect influential observations / bias correction | Jackknife | §6 |
| Handle class imbalance, weighted metrics | Weighted Sampling | §7 |
| Stratified cross-validation without sklearn | Stratified K-Fold | §8 |
| A/B test significance and effect size | Two-Sample Bootstrap | §9 |

---

## When to Reach for Each Method

| Interview signal | Method |
| :--- | :--- |
| "Split without sklearn", "reproducible split" | §1 SRS |
| "Imbalanced classes", "preserve proportions" | §2 Stratified split |
| "Large ordered dataset", "sample a log" | §3 Systematic |
| "Sample by user/session", "grouped data" | §4 Cluster |
| "CI without scipy", "any statistic", "model uncertainty" | §5 Bootstrap |
| "Which observations matter most?", "bias correction" | §6 Jackknife |
| "Minority class", "weighted accuracy/metric" | §7 Weighted |
| "Cross-validation without sklearn", "k-fold" | §8 Stratified K-Fold |
| "A/B test", "compare two groups", "significance" | §9 Two-Sample Bootstrap |

---

> **Interview note:** In a live coding round, use `np.percentile` for bootstrap CIs
> (no scipy needed). For t-critical values, `z = 1.96` is the standard approximation
> for `n >= 30` at 95% confidence. Pre-compute exact values once when `n` is small:
> e.g. `t(0.975, df=49) = 2.010`, `t(0.975, df=99) = 1.984`.

---
## §1 -- Simple Random Sampling (SRS)

**Intuition:** Every unit has an equal probability of selection -- the "lottery" model.
The baseline all other methods compare against. In DS interviews this appears most
often disguised as: *"implement a train/test split without using sklearn."*

**Key properties:**
- Unbiased estimator of the population mean
- Variance decreases as 1/n
- `df.sample(frac=1)` shuffles; slicing gives the split

**Confidence Interval:**

$$\bar{x} \pm z \cdot \frac{s}{\sqrt{n}} \quad (z = 1.96 \text{ for 95\%, valid for } n \geq 30)$$

---

> **Interview prompt:** *"Without using sklearn, implement a function that splits a
> DataFrame into train and test sets given a test fraction. Ensure it is reproducible
> and verify there is no overlap between the two sets."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── Simulate dataset ──────────────────────────────────────────────────────────
N = 10_000
df_pop = pd.DataFrame({
    'user_id': np.arange(N),
    'revenue': rng.exponential(scale=50, size=N).round(2)
})

# ── Core patterns ─────────────────────────────────────────────────────────────
n = 200
sample_pd = df_pop.sample(n=n, replace=False, random_state=42)          # pandas
sample_np  = rng.choice(df_pop['revenue'].values, size=n, replace=False) # NumPy

# ── Interview task: train_test_split from scratch ─────────────────────────────
def train_test_split(df, test_frac=0.2, seed=42):
    """
    Random train/test split without sklearn.
    Shuffles once then slices -- O(n), no index collisions.
    """
    shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_test   = int(len(shuffled) * test_frac)
    test     = shuffled.iloc[:n_test].reset_index(drop=True)
    train    = shuffled.iloc[n_test:].reset_index(drop=True)
    return train, test

train, test = train_test_split(df_pop, test_frac=0.2, seed=42)

# ── Verify: no overlap, correct sizes ────────────────────────────────────────
overlap = set(train['user_id']) & set(test['user_id'])
print(f"Train size : {len(train):,}  ({len(train)/N*100:.0f}%)")
print(f"Test size  : {len(test):,}   ({len(test)/N*100:.0f}%)")
print(f"Overlap    : {len(overlap)} rows  (must be 0)")

# ── Confidence Interval (z-approximation, n >= 30) ───────────────────────────
x     = train['revenue'].values
x_bar = x.mean()
se    = x.std(ddof=1) / np.sqrt(len(x))
z     = 1.96
ci    = (x_bar - z * se, x_bar + z * se)

print(f"\nTrain mean  : {x_bar:.2f}")
print(f"95% CI      : ({ci[0]:.2f}, {ci[1]:.2f})")
print(f"True mean   : {df_pop['revenue'].mean():.2f}")

**Common mistakes:**
- Forgetting `random_state` / `seed` -- split is not reproducible
- Using `df.index` directly after filtering -- duplicated indices cause silent bugs in `.iloc` vs `.loc`; always `reset_index(drop=True)` after a split
- Checking overlap by row content instead of ID -- two different rows can have identical values

---
## §2 -- Stratified Sampling

**Intuition:** Divide the population into non-overlapping strata and sample
independently within each. Guarantees every subgroup is represented in proportion
to its size in the population. In DS interviews this is almost always framed as a
**class-imbalance problem** -- a minority class can disappear entirely from a test
set under plain SRS.

**Key properties:**
- Preserves class proportions in both train and test
- Variance is lower than SRS when strata have different means
- Disproportional allocation = deliberately oversample a minority class

**Confidence Interval:**

$$\bar{x}_{st} = \sum_h W_h \bar{x}_h, \quad SE = \sqrt{\sum_h W_h^2 \frac{s_h^2}{n_h}}$$

where $W_h = N_h / N$ is the stratum weight.

---

> **Interview prompt:** *"You have a dataset with 80/15/5 class imbalance. Implement
> a stratified train/test split without sklearn that preserves those proportions.
> Verify the result."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── Simulate imbalanced classification dataset ────────────────────────────────
N = 5_000
labels   = rng.choice(['class_A', 'class_B', 'class_C'], size=N, p=[0.80, 0.15, 0.05])
df = pd.DataFrame({'feature': rng.normal(0, 1, N), 'label': labels})

print("Population class distribution:")
print(df['label'].value_counts(normalize=True).round(3).to_string())

# ── Interview task: stratified_split from scratch ─────────────────────────────
def stratified_split(df, target_col, test_frac=0.2, seed=42):
    """
    Stratified train/test split preserving class proportions.
    Samples test_frac from each stratum independently.
    """
    train_parts, test_parts = [], []
    for _, grp in df.groupby(target_col):
        shuffled = grp.sample(frac=1, random_state=seed)
        n_test   = int(len(shuffled) * test_frac)
        test_parts.append(shuffled.iloc[:n_test])
        train_parts.append(shuffled.iloc[n_test:])
    train = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    test  = pd.concat(test_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return train, test

train, test = stratified_split(df, target_col='label', test_frac=0.2, seed=42)

# ── Verify proportions are preserved ──────────────────────────────────────────
print("\nClass proportions across splits:")
comparison = pd.DataFrame({
    'population': df['label'].value_counts(normalize=True).round(3),
    'train'     : train['label'].value_counts(normalize=True).round(3),
    'test'      : test['label'].value_counts(normalize=True).round(3),
}).sort_index()
print(comparison)

# ── Disproportional allocation: oversample minority class ─────────────────────
target_n = {'class_A': 200, 'class_B': 200, 'class_C': 200}           # equal allocation
parts = []
for label, grp in df.groupby('label'):
    n     = target_n[label]
    repl  = len(grp) < n                                                # oversample if needed
    parts.append(grp.sample(n=n, replace=repl, random_state=42))
balanced = pd.concat(parts).reset_index(drop=True)

print("\nOversampled training set (equal allocation):")
print(balanced['label'].value_counts().to_string())

# ── Confidence Interval (weighted stratum formula) ────────────────────────────
N_total = len(df)
grp_stats = train.groupby('label').agg(n=('feature','count'),
                                        mean=('feature','mean'),
                                        var=('feature','var'))
grp_stats['N_h'] = df['label'].value_counts()
grp_stats['W_h'] = grp_stats['N_h'] / N_total
strat_mean = (grp_stats['W_h'] * grp_stats['mean']).sum()
strat_var  = (grp_stats['W_h']**2 * grp_stats['var'] / grp_stats['n']).sum()
ci = (strat_mean - 1.96*np.sqrt(strat_var), strat_mean + 1.96*np.sqrt(strat_var))
print(f"\nStratified mean : {strat_mean:.3f}")
print(f"95% CI          : ({ci[0]:.3f}, {ci[1]:.3f})")

**Stratified vs. SRS:**

| | Variance vs SRS | Requires | Best when |
| :--- | :--- | :--- | :--- |
| SRS | Baseline | Nothing | Balanced classes |
| Stratified | Lower | Known class labels | Imbalanced classes, small minorities |

**Common mistakes:**
- Using `df.groupby().apply(lambda x: x.sample(...))` in pandas 3.x -- the groupby key is dropped from the result; use an explicit loop instead
- Forgetting to re-shuffle after `pd.concat` -- train rows are blocked by class, which can cause issues with sequential models
- Oversampling test set -- only oversample the training set; keep test proportions natural

---
## §3 -- Systematic Sampling

**Intuition:** Sort the population by a natural ordering, pick a random start
in [0, k), then select every k-th unit. Simpler than SRS for large ordered data
(one scan, no lookup table) and equally precise when the order is random.
The single failure mode: if the data has a periodic pattern with the same period
as k, the sample is catastrophically biased.

**Step size:** k = N / n (round to nearest integer). **Random start** in [0, k).

**Confidence Interval:** Approximate as SRS when autocorrelation is low.

$$\text{CI} \approx \bar{x} \pm 1.96 \cdot \frac{s}{\sqrt{n}}$$

---

> **Interview prompt:** *"You have a time-ordered log of 1 million server events.
> Implement systematic sampling to draw 10,000 for analysis. Then explain and
> demonstrate the main failure mode."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── Simulate time-ordered event log ──────────────────────────────────────────
N = 1_000_000
df_log = pd.DataFrame({
    'timestamp'   : pd.date_range('2024-01-01', periods=N, freq='s'),
    'response_ms' : rng.exponential(scale=200, size=N).round(1)
})

# ── Systematic sampling ───────────────────────────────────────────────────────
n     = 10_000
k     = N // n                                                           # step = 100
start = int(rng.integers(0, k))                                          # random start in [0, k)
indices    = np.arange(start, N, step=k)[:n]

sample_sys = df_log.iloc[indices].reset_index(drop=True)                 # pandas
sample_np  = df_log['response_ms'].values[indices]                       # NumPy

print(f"Population : {N:,} events")
print(f"Step k     : {k},  Random start: {start},  Sample n: {len(sample_sys):,}")
print(f"Time range : {sample_sys['timestamp'].iloc[0]} -> {sample_sys['timestamp'].iloc[-1]}")

# ── Confidence Interval ───────────────────────────────────────────────────────
x  = sample_sys['response_ms'].values
ci = (x.mean() - 1.96*x.std(ddof=1)/np.sqrt(n),
      x.mean() + 1.96*x.std(ddof=1)/np.sqrt(n))
print(f"\nSample mean : {x.mean():.1f} ms")
print(f"95% CI      : ({ci[0]:.1f}, {ci[1]:.1f}) ms")
print(f"True mean   : {df_log['response_ms'].mean():.1f} ms")

# ── Failure mode: check autocorrelation at lag k ─────────────────────────────
autocorr = pd.Series(df_log['response_ms'].values[:10*k]).autocorr(lag=k)
print(f"\nAutocorrelation at lag k={k}: {autocorr:.4f}")
print("If |autocorr| > 0.1 consider SRS instead")

# ── Demonstrate: periodic data aligned with k ────────────────────────────────
periodic      = np.sin(2 * np.pi * np.arange(N) / k) * 100 + 200
periodic_samp = periodic[indices]
print(f"\nPeriodic signal -- true mean   : {periodic.mean():.1f}")
print(f"Periodic signal -- sample mean : {periodic_samp.mean():.1f}  <- biased")
print(f"Bias magnitude                 : {abs(periodic_samp.mean()-periodic.mean()):.1f}")

**Common mistakes:**
- Not randomizing the starting point -- the sample is no longer probability-based
- Using `df[::k]` without a random start -- always starts at index 0
- Skipping the autocorrelation check -- periodic patterns (e.g. weekly seasonality in daily logs) can completely invalidate the sample

---
## §4 -- Cluster Sampling

**Intuition:** When individuals are naturally grouped (sessions, users, stores),
sampling whole groups is often cheaper than sampling individuals. Select m clusters
at random and use all (or a sub-sample of) their members. The statistical cost:
units within a cluster are similar, so you get less diversity per observation than
SRS -- cluster SE > SRS SE for the same total n.

**Confidence Interval -- between-cluster variance:**

$$\bar{x}_{cl} = \frac{1}{m}\sum_i \bar{x}_i, \quad SE = \frac{s_{\bar{x}}}{\sqrt{m}}, \quad \text{CI} = \bar{x}_{cl} \pm t_{\alpha/2,\,m-1} \cdot SE$$

---

> **Interview prompt:** *"You have clickstream data across 500 app sessions. Sampling
> individual events is expensive. Implement cluster sampling by session and show
> numerically why the variance is higher than SRS of the same total n."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── Simulate session-level event data ─────────────────────────────────────────
n_sessions = 500
sizes      = rng.integers(20, 80, size=n_sessions)                       # variable session size
session_ids = np.repeat(np.arange(n_sessions), sizes)
df_events  = pd.DataFrame({
    'session_id' : session_ids,
    'duration_s' : rng.exponential(30, size=len(session_ids)).round(1),
    'converted'  : rng.binomial(1, 0.08, size=len(session_ids))
})
print(f"Total events : {len(df_events):,} across {n_sessions} sessions")

# ── One-stage: sample whole sessions ──────────────────────────────────────────
m        = 50
selected = rng.choice(n_sessions, size=m, replace=False)
one_stage = df_events[df_events['session_id'].isin(selected)].copy()
print(f"\nOne-stage  : {m} sessions, {len(one_stage):,} events")

# ── Two-stage: subsample events within selected sessions ──────────────────────
parts = []
for sid in selected:
    grp = df_events[df_events['session_id'] == sid]
    parts.append(grp.sample(frac=0.5, random_state=42))
two_stage = pd.concat(parts, ignore_index=True)
print(f"Two-stage  : {m} sessions, {len(two_stage):,} events (50% within each session)")

# ── Confidence Interval (between-cluster variance) ────────────────────────────
cluster_means = one_stage.groupby('session_id')['duration_s'].mean()
c_bar  = float(cluster_means.mean())
c_se   = cluster_means.std(ddof=1) / np.sqrt(m)
t_crit = 2.010                                                            # t(0.975, df=49)
ci     = (c_bar - t_crit * c_se, c_bar + t_crit * c_se)

print(f"\nCluster mean : {c_bar:.2f} s")
print(f"95% CI       : ({ci[0]:.2f}, {ci[1]:.2f}) s")
print(f"True mean    : {df_events['duration_s'].mean():.2f} s")

# ── Variance comparison: cluster vs SRS of the same total n ──────────────────
srs_n = len(one_stage)
srs_ests, cluster_ests = [], []
rng2 = np.random.default_rng(99)
for _ in range(300):
    srs_ests.append(df_events['duration_s'].sample(n=srs_n, random_state=None).mean())
    sel = rng2.choice(n_sessions, size=m, replace=False)
    cm  = df_events[df_events['session_id'].isin(sel)].groupby('session_id')['duration_s'].mean()
    cluster_ests.append(cm.mean())

print(f"\nVariance comparison (300 resamples, same total n={srs_n:,}):")
print(f"  SRS SE     : {np.std(srs_ests, ddof=1):.4f}")
print(f"  Cluster SE : {np.std(cluster_ests, ddof=1):.4f}  <- larger due to intra-session similarity")
print("  Trade-off  : cluster sampling is cheaper to collect but less statistically efficient")

**Cluster vs. Stratified -- the key distinction:**

| | Effect on variance | Within-group units are... | Use when |
| :--- | :--- | :--- | :--- |
| Stratified | Reduces variance | Heterogeneous (spread out) | Precision is the priority |
| Cluster | Increases variance | Homogeneous (similar) | Collection cost is the constraint |

**Common mistakes:**
- Confusing cluster and stratified -- they have **opposite** effects on variance
- Using only m = 2 or 3 clusters -- t(0.975, df=1) = 12.7, giving an extremely wide CI; aim for m >= 10
- Forgetting that the CI uses **cluster-level means** as the unit, not individual events

---
## §5 -- Bootstrap Resampling

**Intuition:** Your sample is the best proxy for the population you have. Draw
B samples *with replacement* from your data, compute the statistic on each --
the spread of those B estimates approximates the true sampling distribution.
Works for **any statistic**, requires no distributional assumptions, and is the
standard approach for model performance uncertainty in industry.

**Algorithm:** resample with replacement B times -> compute stat -> take percentiles.

**Confidence Interval (percentile method -- no scipy needed):**

$$\text{CI} = [\hat{\theta}^*_{\alpha/2},\; \hat{\theta}^*_{1-\alpha/2}]$$

where $\hat{\theta}^*$ are the sorted bootstrap estimates and `np.percentile` does the work.

---

> **Interview prompt (3 parts):**
> *(a) Implement a bootstrap CI for the median without scipy.*
> *(b) Use bootstrap to get a 95% CI on a classifier's test AUC.*
> *(c) Preview: bootstrap CI on the difference between two groups.*

In [ ]:
import numpy as np
import pandas as pd

# ── (a) General bootstrap_ci: works for any statistic ────────────────────────
def bootstrap_ci(data, stat_fn, B=2000, confidence=0.95, seed=42):
    """
    Bootstrap confidence interval for any statistic.
    stat_fn : callable accepting a 1-D array, returning a scalar.
    Uses the percentile method -- no scipy needed.
    """
    rng_b = np.random.default_rng(seed)
    n     = len(data)
    # Vectorised resampling: draw B samples in one shot
    idx        = rng_b.integers(0, n, size=(B, n))
    boot_stats = np.array([stat_fn(data[i]) for i in idx])
    alpha      = 1 - confidence
    lo, hi     = np.percentile(boot_stats, [100*alpha/2, 100*(1-alpha/2)])
    return lo, hi, boot_stats

rng  = np.random.default_rng(42)
data = rng.exponential(scale=50, size=300)                               # skewed -- bootstrap shines here
B    = 2000

stats_to_test = {
    'mean'        : np.mean,
    'median'      : np.median,
    'std'         : np.std,
    '90th pctile' : lambda x: np.percentile(x, 90),
}

print(f"{'Statistic':<14} {'Observed':>10} {'95% CI':>26}  {'Bootstrap SE':>14}")
print("-" * 68)
for name, fn in stats_to_test.items():
    obs        = fn(data)
    lo, hi, bs = bootstrap_ci(data, fn, B=B)
    print(f"{name:<14} {obs:>10.3f}   ({lo:.3f}, {hi:.3f})  {bs.std(ddof=1):>14.3f}")

In [ ]:
import numpy as np
import pandas as pd

# ── (b) Bootstrap CI on model evaluation metrics ──────────────────────────────
rng = np.random.default_rng(42)
n   = 1000
y_true = rng.binomial(1, 0.3, size=n)
y_prob = np.clip(y_true * 0.6 + rng.normal(0, 0.25, n), 0, 1)
y_pred = (y_prob >= 0.5).astype(int)

def accuracy(yt, yp):
    return float(np.mean(yt == yp))

def roc_auc(yt, yp):
    """Trapezoidal AUC (pure NumPy, no sklearn)."""
    order = np.argsort(yp)[::-1]
    yt_s  = yt[order]; yp_s = yp[order]
    pos   = yt.sum();  neg  = (1 - yt).sum()
    tp = np.cumsum(yt_s)   / pos
    fp = np.cumsum(1-yt_s) / neg
    tp = np.concatenate([[0], tp]); fp = np.concatenate([[0], fp])
    # Manual trapezoid (np.trapz renamed in NumPy 2.x)
    return float(np.sum((fp[1:] - fp[:-1]) * (tp[1:] + tp[:-1]) / 2))

def bootstrap_metric_ci(y_true, y_score, metric_fn, B=2000, confidence=0.95, seed=42):
    """Bootstrap CI for any model metric. metric_fn(y_true, y_score) -> scalar."""
    rng_b  = np.random.default_rng(seed)
    n      = len(y_true)
    boot   = np.array([
        metric_fn(y_true[idx], y_score[idx])
        for idx in rng_b.integers(0, n, size=(B, n))
    ])
    alpha  = 1 - confidence
    lo, hi = np.percentile(boot, [100*alpha/2, 100*(1-alpha/2)])
    return float(metric_fn(y_true, y_score)), lo, hi, boot

acc, acc_lo, acc_hi, _ = bootstrap_metric_ci(y_true, y_pred, accuracy)
auc, auc_lo, auc_hi, _ = bootstrap_metric_ci(y_true, y_prob, roc_auc)

print("Bootstrap CI on model performance  (n=1000, B=2000)")
print(f"  {'Metric':<12} {'Point est':>10} {'95% CI':>26}")
print(f"  {'Accuracy':<12} {acc:>10.4f}   ({acc_lo:.4f}, {acc_hi:.4f})")
print(f"  {'AUC':<12} {auc:>10.4f}   ({auc_lo:.4f}, {auc_hi:.4f})")
print("\nThe CI shows the uncertainty in the metric across hypothetical test sets.")
print("A single test-set score is just one draw from this distribution.")

In [ ]:
import numpy as np
import pandas as pd

# ── (c) Bootstrap CI on the difference (preview of §9) ───────────────────────
rng = np.random.default_rng(42)
group_a = rng.normal(50, 10, 300)                                        # control
group_b = rng.normal(53, 10, 300)                                        # treatment (+3 units)

def bootstrap_diff_ci(a, b, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """Bootstrap CI for stat_fn(b) - stat_fn(a)."""
    rng_b  = np.random.default_rng(seed)
    diffs  = np.array([
        stat_fn(rng_b.choice(b, len(b), replace=True)) -
        stat_fn(rng_b.choice(a, len(a), replace=True))
        for _ in range(B)
    ])
    alpha  = 1 - confidence
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return float(stat_fn(b) - stat_fn(a)), lo, hi

obs_diff, lo, hi = bootstrap_diff_ci(group_a, group_b)
print(f"Observed difference (B - A) : {obs_diff:.3f}")
print(f"95% CI on difference        : ({lo:.3f}, {hi:.3f})")
print(f"Excludes zero (significant) : {lo > 0}")
print("\nFull two-sample test with p-value and permutation alternative in §9.")

**Bootstrap CI methods compared:**

| Method | Scipy needed? | Handles skew? | When to use |
| :--- | :--- | :--- | :--- |
| Percentile | No -- `np.percentile` | No | Quick CI in interviews |
| Basic / pivot | No | Partially | Better for biased bootstrap |
| BCa | Yes (`scipy.stats.bootstrap`) | Yes | Production code |

**Common mistakes:**
- B < 1000 resamples -- CIs are unstable; B = 2000 is the minimum for interviews
- Bootstrapping time-series with plain row resampling -- violates independence; use block bootstrap
- Using the same seed for both groups in a two-sample bootstrap -- they must be sampled independently

---
## §6 -- Jackknife Resampling

**Intuition:** Leave one observation out at a time, recompute the statistic on
the remaining n-1 -- repeat for every observation. The n leave-one-out estimates
reveal how much each individual observation influences the result. Use jackknife
for (a) bias correction and (b) finding influential observations.
Only works reliably for **smooth** statistics (mean, ratio) -- fails for median
and quantiles. Bootstrap is the safer default; jackknife is the specialist tool.

**Jackknife bias correction:**

$$\text{bias} = (n-1)(\bar{\theta}_{(.)} - \hat{\theta}), \quad \hat{\theta}_{jack} = n\hat{\theta} - (n-1)\bar{\theta}_{(.)}$$

**Confidence Interval:**

$$SE_{jack} = \sqrt{\frac{n-1}{n}\sum_i(\hat{\theta}_{(i)} - \bar{\theta}_{(.)})^2}, \quad \text{CI} = \hat{\theta} \pm t_{\alpha/2,n-1} \cdot SE_{jack}$$

---

> **Interview prompt:** *"Implement leave-one-out jackknife to identify the
> observations that most influence the sample mean. Then show why jackknife
> gives wrong answers for the median."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Dataset with known outliers to make influential-obs detection interesting
data = np.concatenate([rng.normal(50, 10, 95), [90, 85, 80, 15, 20]])
n    = len(data)

# ── Jackknife leave-one-out estimates ────────────────────────────────────────
jk_stats   = np.array([np.mean(np.delete(data, i)) for i in range(n)])  # NumPy
# Pandas equivalent:
s = pd.Series(data)
jk_stats_pd = np.array([s.drop(index=i).mean() for i in range(n)])

# ── Bias estimation and correction ───────────────────────────────────────────
theta_hat  = data.mean()
theta_bar  = jk_stats.mean()
bias_est   = (n - 1) * (theta_bar - theta_hat)
theta_jack = n * theta_hat - (n - 1) * theta_bar                        # bias-corrected

# ── Jackknife SE and CI ───────────────────────────────────────────────────────
jk_var = ((n - 1) / n) * np.sum((jk_stats - theta_bar) ** 2)
jk_se  = np.sqrt(jk_var)
t_crit = 1.984                                                           # t(0.975, df=99)
ci     = (theta_hat - t_crit * jk_se, theta_hat + t_crit * jk_se)

print(f"Observed mean       : {theta_hat:.4f}")
print(f"Jackknife bias      : {bias_est:.6f}  (small = unbiased estimator)")
print(f"Bias-corrected mean : {theta_jack:.4f}")
print(f"Jackknife SE        : {jk_se:.4f}")
print(f"95% CI              : ({ci[0]:.3f}, {ci[1]:.3f})")

# ── Interview task: detect most influential observations ──────────────────────
influence = np.abs(jk_stats - theta_hat)                                 # per-obs influence
top_k     = 5
top_idx   = np.argsort(influence)[::-1][:top_k]

print(f"\nTop {top_k} most influential observations:")
print(f"  {'Rank':<6} {'Index':>6} {'Value':>8} {'LOO mean':>10} {'Influence':>12}")
for rank, idx in enumerate(top_idx):
    print(f"  {rank+1:<6} {idx:>6} {data[idx]:>8.1f} {jk_stats[idx]:>10.4f} {influence[idx]:>12.4f}")

# ── When jackknife fails: the median ─────────────────────────────────────────
jk_medians = np.array([np.median(np.delete(data, i)) for i in range(n)])
jk_se_med  = np.sqrt(((n-1)/n) * np.sum((jk_medians - jk_medians.mean())**2))
boot_se_med = np.std([np.median(rng.choice(data, n, replace=True)) for _ in range(2000)], ddof=1)
print(f"\nJackknife SE for median : {jk_se_med:.4f}  <- wrong (non-smooth statistic)")
print(f"Bootstrap SE for median : {boot_se_med:.4f}  <- correct")
print("Rule: use jackknife for mean/ratio; use bootstrap for median/quantile/custom metrics")

**Jackknife vs. Bootstrap:**

| | Resamples | Smooth stats only? | Bias correction? | Cost |
| :--- | :--- | :--- | :--- | :--- |
| Jackknife | n (exact) | Yes | Yes | O(n) |
| Bootstrap | B >= 2000 | No | Partial (BCa) | O(B * n) |

**Common mistakes:**
- Applying jackknife to the median, quantiles, or max -- produces wrong SE estimates
- Confusing jackknife with LOOCV -- same leave-one-out structure, different purpose (LOOCV measures prediction error; jackknife measures statistical variance)
- Interpreting high jackknife influence as "delete this observation" -- it means investigate it, not remove it

---
## §7 -- Weighted Sampling

**Intuition:** When observations have unequal importance -- due to class imbalance,
oversampling, or survey design -- weights restore representativeness. Each observation
gets weight proportional to 1 / (selection probability). In DS interviews this almost
always appears as **class imbalance**: the minority class needs more weight so the
model does not ignore it.

**Inverse class frequency weight:** $w_i = 1 / \text{count}(\text{class of } i)$

**Confidence Interval using Kish effective sample size:**

$$\bar{x}_w = \frac{\sum w_i x_i}{\sum w_i}, \quad n_{eff} = \frac{(\sum w_i)^2}{\sum w_i^2}, \quad SE_w = \sqrt{\frac{s_w^2}{n_{eff}}}$$

Using $n_{eff}$ instead of raw $n$ correctly accounts for high variance in weights.

---

> **Interview prompt:** *"You have a binary dataset with 95/5 class imbalance.
> Implement (a) weighted sampling to create a balanced training set, and (b)
> inverse-frequency weighting to compute a balanced accuracy metric. Verify the
> class balance after sampling."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── Simulate imbalanced binary dataset ───────────────────────────────────────
N      = 5_000
labels = rng.choice([0, 1], size=N, p=[0.95, 0.05])
df     = pd.DataFrame({
    'feature' : rng.normal(labels * 0.5, 1, size=N),
    'label'   : labels
})
print("Original class distribution:")
print(df['label'].value_counts().to_string())

# ── Inverse class frequency weights ──────────────────────────────────────────
class_counts         = df['label'].value_counts()
df['sample_weight']  = df['label'].map(1.0 / class_counts)

# ── (a) Weighted sampling to balance training set ────────────────────────────
# replace=True required when minority class is smaller than requested n
balanced_pd = df.sample(n=1_000, weights='sample_weight', replace=True, random_state=42)

# NumPy version (weights must sum to 1 explicitly)
w_norm     = df['sample_weight'].values / df['sample_weight'].values.sum()
idx_np     = rng.choice(len(df), size=1_000, replace=True, p=w_norm)
balanced_np = df.iloc[idx_np]

# ── Verify class balance ──────────────────────────────────────────────────────
print("\nBalanced sample proportions (pandas):")
print(balanced_pd['label'].value_counts(normalize=True).round(3).to_string())
print("\nBalanced sample proportions (NumPy):")
print(pd.Series(balanced_np['label'].values).value_counts(normalize=True).round(3).to_string())

# ── (b) Inverse-frequency weighted accuracy ───────────────────────────────────
y_true = df['label'].values
y_pred = (df['feature'].values > 0.25).astype(int)                      # simple threshold
w      = df['sample_weight'].values

unweighted_acc = float(np.mean(y_true == y_pred))
weighted_acc   = float(np.average(y_true == y_pred, weights=w))         # balanced accuracy

print(f"\nUnweighted accuracy : {unweighted_acc:.4f}  <- dominated by majority class (easy 0s)")
print(f"Weighted accuracy   : {weighted_acc:.4f}  <- balanced; penalises minority errors equally")

# ── Confidence Interval using Kish effective n ───────────────────────────────
vals   = df['feature'].values
w_mean = float(np.average(vals, weights=w))
w_var  = float(np.average((vals - w_mean)**2, weights=w))
n_eff  = w.sum()**2 / (w**2).sum()                                      # Kish formula
w_se   = np.sqrt(w_var / n_eff)
ci     = (w_mean - 1.96 * w_se, w_mean + 1.96 * w_se)

print(f"\nKish effective n    : {n_eff:.0f}  (vs raw n = {N})")
print(f"Weighted mean       : {w_mean:.4f}")
print(f"95% CI              : ({ci[0]:.4f}, {ci[1]:.4f})")
print("Using n_eff prevents overconfident CIs when weights are highly variable")

**Common mistakes:**
- `df.sample(weights=col, replace=False)` fails when weights are very unequal and the sample is large -- use `replace=True` for oversampling
- `np.random.choice(p=weights)` requires weights that already sum to 1 -- pandas normalizes automatically, NumPy does not
- Using raw n in the weighted SE formula -- always use Kish n_eff; otherwise the CI is too narrow when weights vary a lot
- Oversampling the test set -- only oversample the training set; evaluate on the natural class distribution

---
## §8 -- Stratified K-Fold Cross-Validation

**Intuition:** K-fold splits the data into k equal folds, trains on k-1 folds, and
validates on the remaining one -- rotating until every fold has been the validation
set. **Stratified** k-fold ensures each fold has the same class proportions as the
full dataset. Without stratification, a small minority class can vanish entirely from
a validation fold, making the CV score meaningless for that class.

**Why this appears in DS interviews:** It is the most common cross-validation pattern
in production ML, yet sklearn's `StratifiedKFold` hides all the logic. Interviewers
use "implement it from scratch" to test whether you understand what it is actually doing.

**Core logic:** Assign fold IDs round-robin **within each class**, then rotate.

---

> **Interview prompt:** *"Implement stratified k-fold cross-validation without sklearn.
> For every fold, verify the class proportions in the validation set match the full
> dataset. Show what goes wrong without stratification on an imbalanced dataset."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── Simulate imbalanced 3-class dataset ───────────────────────────────────────
N      = 1_000
labels = rng.choice([0, 1, 2], size=N, p=[0.70, 0.20, 0.10])
df     = pd.DataFrame(rng.normal(size=(N, 3)), columns=['f1', 'f2', 'f3'])
df['label'] = labels
pop_props = df['label'].value_counts(normalize=True).sort_index()
print("Population class proportions:")
print(pop_props.round(3).to_string())

# ── Stratified K-Fold from scratch ────────────────────────────────────────────
def stratified_kfold(df, target_col, k=5, seed=42):
    """
    Yields (train_indices, val_indices) for k stratified folds.
    Core idea: assign fold IDs round-robin within each class,
    then rotate which fold is the validation set.
    """
    rng_k        = np.random.default_rng(seed)
    fold_ids     = np.full(len(df), -1, dtype=int)
    for _, grp in df.groupby(target_col):
        idx = grp.index.values.copy()
        rng_k.shuffle(idx)                                               # shuffle within class
        for pos, i in enumerate(idx):
            fold_ids[i] = pos % k                                        # assign round-robin
    for fold in range(k):
        val_idx   = np.where(fold_ids == fold)[0]
        train_idx = np.where(fold_ids != fold)[0]
        yield train_idx, val_idx

# ── Run and verify proportions every fold ─────────────────────────────────────
k = 5
print(f"\nStratified {k}-Fold -- validation class proportions:")
header = f"{'Fold':<6} {'Train':>8} {'Val':>6}  {'cls_0':>8} {'cls_1':>8} {'cls_2':>8}"
print(header); print("-" * len(header))

for fold, (train_idx, val_idx) in enumerate(stratified_kfold(df, 'label', k=k)):
    vp = df.iloc[val_idx]['label'].value_counts(normalize=True).sort_index()
    print(f"{fold:<6} {len(train_idx):>8} {len(val_idx):>6}  "
          f"{vp.get(0,0):>8.3f} {vp.get(1,0):>8.3f} {vp.get(2,0):>8.3f}")
print(f"{'Pop':.<6} {'':>8} {'':>6}  "
      f"{pop_props[0]:>8.3f} {pop_props[1]:>8.3f} {pop_props[2]:>8.3f}")

# ── What goes wrong without stratification ─────────────────────────────────────
print(f"\nNon-stratified {k}-Fold -- class_2 proportion per fold (target {pop_props[2]:.3f}):")
all_idx   = rng.permutation(N)
fold_size = N // k
for fold in range(k):
    val_idx = all_idx[fold*fold_size:(fold+1)*fold_size]
    vp      = df.iloc[val_idx]['label'].value_counts(normalize=True).sort_index()
    prop2   = vp.get(2, 0)
    flag    = '  <- off target' if abs(prop2 - pop_props[2]) > 0.02 else ''
    print(f"  Fold {fold}: class_2 = {prop2:.3f}{flag}")

**Common mistakes:**
- Shuffling the full dataset once and then slicing -- does NOT stratify; the shuffle mixes classes but does not control fold composition
- Data leakage: applying preprocessing (scaling, encoding) on the full dataset before splitting -- always fit transformers on the training fold only
- Using integer index arithmetic on a DataFrame without `reset_index()` -- gaps in the index after filtering cause off-by-one errors in `iloc` vs `loc`
- Using the same fold split for both hyperparameter tuning and final evaluation -- use nested CV or a held-out test set

---
## §9 -- Two-Sample Bootstrap & Permutation Test

**Intuition (bootstrap):** To get a CI on the *difference* between two groups,
resample each group independently with replacement, compute the difference of the
statistic on each pair of resamples -- the distribution of those differences is
the bootstrap sampling distribution of the difference.

**Intuition (permutation test):** Under the null hypothesis that the two groups
are exchangeable, shuffling which observations are labelled A vs B should produce
a difference close to zero. Compute this shuffle-difference B times to build the
null distribution, then check where the observed difference sits.

**CI on the difference:**

$$\text{CI} = [\delta^*_{\alpha/2},\; \delta^*_{1-\alpha/2}] \quad \text{where } \delta^* = \hat{\theta}_B^* - \hat{\theta}_A^*$$

**Bootstrap p-value (one-sided):** $p = P(\delta^* \leq 0)$

**Permutation p-value (two-sided):** $p = P(|\delta_{null}| \geq |\delta_{obs}|)$

---

> **Interview prompt:** *"You ran an A/B test. Group A has 500 users (8% conversion),
> Group B has 500 users. Without scipy, determine whether the difference in conversion
> rates is statistically significant. Report a 95% CI on the difference and a p-value."*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── A/B test: binary conversion data ─────────────────────────────────────────
n_a, n_b = 500, 500
conv_a    = rng.binomial(1, 0.08, n_a)                                  # control:   8% rate
conv_b    = rng.binomial(1, 0.11, n_b)                                  # treatment: 11% rate
obs_diff  = conv_b.mean() - conv_a.mean()

print(f"Group A conversion : {conv_a.mean():.4f}  (n={n_a})")
print(f"Group B conversion : {conv_b.mean():.4f}  (n={n_b})")
print(f"Observed diff B-A  : {obs_diff:+.4f}")

# ── Bootstrap CI on the difference (vectorised) ───────────────────────────────
def bootstrap_diff_ci(a, b, stat_fn=np.mean, B=10_000, confidence=0.95, seed=42):
    """
    Bootstrap CI for stat_fn(b) - stat_fn(a).
    Resamples each group independently -- vectorised for speed.
    """
    rng_b    = np.random.default_rng(seed)
    idx_a    = rng_b.integers(0, len(a), size=(B, len(a)))
    idx_b    = rng_b.integers(0, len(b), size=(B, len(b)))
    boot_d   = b[idx_b].mean(axis=1) - a[idx_a].mean(axis=1)
    alpha    = 1 - confidence
    lo, hi   = np.percentile(boot_d, [100*alpha/2, 100*(1-alpha/2)])
    p_one    = float(np.mean(boot_d <= 0))                              # P(B <= A), one-sided
    return lo, hi, p_one, boot_d

lo, hi, boot_p, boot_d = bootstrap_diff_ci(conv_a, conv_b, B=10_000)
print(f"\n95% CI on difference : ({lo:.4f}, {hi:.4f})")
print(f"CI excludes zero     : {lo > 0}  (significant at 5%)")
print(f"Bootstrap p (one-sided): {boot_p:.4f}")

# ── Permutation test (two-sided, exact null) ──────────────────────────────────
def permutation_test(a, b, B=10_000, seed=42):
    """
    Tests H0: mean(a) == mean(b) by permuting group labels.
    Returns two-sided p-value and the null distribution.
    """
    rng_p    = np.random.default_rng(seed)
    combined = np.concatenate([a, b])
    n_a      = len(a)
    obs      = b.mean() - a.mean()
    # Vectorised permutation matrix
    perms     = np.array([rng_p.permutation(len(combined)) for _ in range(B)])
    null_d    = (combined[perms[:, n_a:]].mean(axis=1) -
                 combined[perms[:, :n_a]].mean(axis=1))
    p_two    = float(np.mean(np.abs(null_d) >= np.abs(obs)))
    return p_two, null_d

perm_p, null_d = permutation_test(conv_a, conv_b, B=10_000)
print(f"Permutation p (two-sided): {perm_p:.4f}")

# ── Side-by-side comparison ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'Method':<28} {'Result'}")
print("=" * 60)
print(f"{'Bootstrap CI (95%)':<28} ({lo:.4f}, {hi:.4f})")
print(f"{'Bootstrap p (one-sided)':<28} {boot_p:.4f}")
print(f"{'Permutation p (two-sided)':<28} {perm_p:.4f}")
print("=" * 60)
print("\nBootstrap  --> use when you need the EFFECT SIZE and its uncertainty")
print("Permutation --> use when you need a p-value under an exact null hypothesis")

**Bootstrap vs. Permutation -- when to use each:**

| | Bootstrap CI | Permutation test |
| :--- | :--- | :--- |
| Primary output | Effect size + uncertainty | p-value |
| Null hypothesis | Not required | Exact exchangeability |
| Works for any statistic | Yes | Yes |
| Best for | Estimation, effect size reporting | Hypothesis testing |

**Common mistakes:**
- Using a one-sided p-value when the test is not directional -- report two-sided unless the alternative was specified beforehand
- Reporting only a p-value without a CI -- the CI tells you whether the effect is practically meaningful, not just statistically significant
- Resampling both groups from the combined pool in the bootstrap -- groups must be resampled **independently** (combine-and-resample is the permutation test, not the bootstrap)
- Too few permutations -- B = 10,000 gives a p-value stable to ~0.003; B = 1,000 is too noisy for a final answer

---
## §10 -- Interview Decision Guide

```
What is the interviewer asking you to do?

+-- "Split data" / "train-test"
|   +-- Plain random split                          -> §1 train_test_split
|   +-- "Preserve class balance" / imbalanced data  -> §2 stratified_split
|   +-- "Large ordered data" / log files            -> §3 Systematic sampling
|   +-- "Grouped data" / sessions / users           -> §4 Cluster sampling
|
+-- "Cross-validation" / "k-fold"
|   +-- Any mention of class imbalance              -> §8 Stratified k-fold
|
+-- "Confidence interval" / "uncertainty"
|   +-- CI for a standard stat, large n             -> z = 1.96 approximation
|   +-- CI for any stat, unknown distribution       -> §5 Bootstrap (percentile)
|   +-- "Which obs are influential?" / bias         -> §6 Jackknife
|   +-- Imbalanced / survey data                    -> §7 Weighted CI (Kish n_eff)
|
+-- "A/B test" / "compare two groups" / "significance"
    +-- Need a CI on the difference                 -> §9 Bootstrap diff CI
    +-- Need a p-value                              -> §9 Permutation test
    +-- Need both                                   -> §9 Run both, report both
```

**Full comparison:**

| Method | Core pandas/NumPy pattern | CI approach | Scipy? |
| :--- | :--- | :--- | :--- |
| SRS | `df.sample(frac=1)` + slice | z * se | No |
| Stratified split | `groupby` + `sample` per group | Weighted stratum variance | No |
| Systematic | `df.iloc[start::k]` | ~SRS z-approx | No |
| Cluster | `isin(selected_ids)` | Between-cluster SE + t | No |
| Bootstrap | `rng.integers(0,n,(B,n))` + `percentile` | `np.percentile` | No |
| Jackknife | `np.delete(data, i)` per i | Jackknife SE + t | No |
| Weighted | `df.sample(weights=col)` | Weighted SE + Kish n_eff | No |
| Stratified k-fold | `groupby` + round-robin fold IDs | Not applicable | No |
| Two-sample bootstrap | Independent resample each group | `np.percentile` on diffs | No |
| Permutation test | `rng.permutation` + shuffle labels | p-value from null dist | No |